# 🩺 Esteira de Aprendizado de Máquina — Predição de Diabetes

**Dataset:** Pima Indians Diabetes Database  
**Fonte:** [Kaggle](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)  
**Problema:** Classificação binária — prever se um paciente tem diabetes (1) ou não (0)  

---
### Etapas da Esteira
1. Carregamento e exploração dos dados
2. Estatísticas descritivas
3. Transformações nas colunas
4. Transformações nas linhas
5. Divisão em treino, validação e teste
6. Normalização
7. Treinamento do modelo
8. Avaliação (matriz de confusão e acurácia)
9. Predição com o modelo implantado

## 0. Importação das Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

print('Bibliotecas importadas com sucesso!')

## 1. Carregamento dos Dados

O dataset contém dados clínicos de pacientes do sexo feminino com herança indígena Pima.  
O objetivo é prever a presença de diabetes com base em variáveis como glicose, IMC e idade.

In [ ]:
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'

colunas = [
    'Gestacoes',
    'Glicose',
    'PressaoSanguinea',
    'EspessuraPele',
    'Insulina',
    'IMC',
    'FuncaoPedigree',
    'Idade',
    'Resultado'
]

df = pd.read_csv(url, names=colunas)

print(f'Shape do dataset: {df.shape}')
print(f'\nPrimeiras linhas:')
df.head()

## 2. Estatísticas Descritivas

In [ ]:
print('=== Informações Gerais ===')
df.info()

In [ ]:
print('=== Estatísticas Descritivas ===')
df.describe().round(2)

In [ ]:
print('=== Distribuição da variável alvo (Resultado) ===')
print(df['Resultado'].value_counts())
print(f'\nPorcentagem de pacientes SEM diabetes: {(df["Resultado"]==0).mean()*100:.1f}%')
print(f'Porcentagem de pacientes COM diabetes:  {(df["Resultado"]==1).mean()*100:.1f}%')

In [ ]:
print('=== Valores Nulos por Coluna ===')
print(df.isnull().sum())

In [ ]:
# Distribuição de todas as variáveis
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(colunas):
    axes[i].hist(df[col], bins=20, color='steelblue', edgecolor='white')
    axes[i].set_title(col, fontsize=11)

plt.suptitle('Distribuição das Variáveis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Mapa de correlação
plt.figure(figsize=(10, 7))
sns.heatmap(
    df.corr(),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    square=True,
    linewidths=0.5
)
plt.title('Mapa de Correlação entre as Variáveis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Transformações nas Colunas

Colunas como `Glicose`, `PressaoSanguinea`, `EspessuraPele`, `Insulina` e `IMC` possuem valor 0,  
o que é biologicamente impossível — esses zeros representam **dados ausentes**.  

**Transformação aplicada:** substituição dos zeros pela **mediana** de cada coluna  
*(a mediana é mais robusta a outliers do que a média)*

In [ ]:
df_limpo = df.copy()

colunas_com_zeros_invalidos = ['Glicose', 'PressaoSanguinea', 'EspessuraPele', 'Insulina', 'IMC']

for col in colunas_com_zeros_invalidos:
    qtd_zeros = (df_limpo[col] == 0).sum()
    mediana = df_limpo[df_limpo[col] != 0][col].median()
    df_limpo[col] = df_limpo[col].replace(0, mediana)
    print(f'{col}: {qtd_zeros} zeros substituídos pela mediana ({mediana:.2f})')

print('\nTransformação nas colunas concluída!')

## 4. Transformações nas Linhas

**Transformação aplicada:** remoção de linhas duplicadas e de outliers extremos  
usando o método **IQR (Interquartile Range)** nas colunas mais sensíveis.

In [ ]:
# Etapa 1: Remover duplicatas
linhas_antes = len(df_limpo)
df_limpo = df_limpo.drop_duplicates()
duplicatas_removidas = linhas_antes - len(df_limpo)
print(f'Duplicatas removidas: {duplicatas_removidas}')

# Etapa 2: Remover outliers extremos via IQR
colunas_outlier = ['Glicose', 'IMC', 'Insulina', 'PressaoSanguinea']

linhas_antes = len(df_limpo)

for col in colunas_outlier:
    Q1 = df_limpo[col].quantile(0.25)
    Q3 = df_limpo[col].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 3 * IQR
    limite_superior = Q3 + 3 * IQR
    df_limpo = df_limpo[
        (df_limpo[col] >= limite_inferior) & (df_limpo[col] <= limite_superior)
    ]

outliers_removidos = linhas_antes - len(df_limpo)
df_limpo = df_limpo.reset_index(drop=True)

print(f'Linhas com outliers extremos removidas: {outliers_removidos}')
print(f'Shape final após transformações: {df_limpo.shape}')

## 5. Divisão em Treino, Validação e Teste

Proporção adotada: **60% treino | 20% validação | 20% teste**

In [ ]:
X = df_limpo.drop('Resultado', axis=1)
y = df_limpo['Resultado']

# Primeira divisão: 80% treino+validação / 20% teste
X_temp, X_teste, y_temp, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Segunda divisão: 60% treino / 20% validação (do total)
X_treino, X_val, y_treino, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f'Conjunto de Treino:    {X_treino.shape[0]} amostras ({X_treino.shape[0]/len(X)*100:.0f}%)')
print(f'Conjunto de Validação: {X_val.shape[0]} amostras ({X_val.shape[0]/len(X)*100:.0f}%)')
print(f'Conjunto de Teste:     {X_teste.shape[0]} amostras ({X_teste.shape[0]/len(X)*100:.0f}%)')

## 6. Normalização dos Dados

Aplicamos `StandardScaler` para padronizar as features.  
O scaler é **ajustado apenas no treino** e aplicado nos demais conjuntos — evitando *data leakage*.

In [ ]:
scaler = StandardScaler()

X_treino_scaled = scaler.fit_transform(X_treino)
X_val_scaled    = scaler.transform(X_val)
X_teste_scaled  = scaler.transform(X_teste)

print('Normalização aplicada com sucesso!')
print(f'Média após escalar (treino):         {X_treino_scaled.mean():.4f}')
print(f'Desvio padrão após escalar (treino): {X_treino_scaled.std():.4f}')

## 7. Treinamento do Modelo

Modelo escolhido: **Random Forest Classifier**

> Random Forest é um ensemble de múltiplas árvores de decisão.  
> É robusto, lida bem com dados clínicos e não requer features perfeitamente distribuídas.

O conjunto de **validação** é usado para monitorar o desempenho antes da avaliação final.

In [ ]:
modelo = RandomForestClassifier(
    n_estimators=150,
    max_depth=8,
    min_samples_split=5,
    random_state=42
)

modelo.fit(X_treino_scaled, y_treino)

print('Modelo treinado com sucesso!')

# Avaliação no conjunto de validação
y_pred_val = modelo.predict(X_val_scaled)
acuracia_val = accuracy_score(y_val, y_pred_val)
print(f'\nAcurácia no conjunto de VALIDAÇÃO: {acuracia_val*100:.2f}%')

## 8. Avaliação do Modelo — Conjunto de Teste

### 8.1 Acurácia e Relatório de Classificação

In [ ]:
y_pred_teste = modelo.predict(X_teste_scaled)

acuracia_teste = accuracy_score(y_teste, y_pred_teste)
print(f'Acurácia no conjunto de TESTE: {acuracia_teste*100:.2f}%')
print()
print('=== Relatório de Classificação ===')
print(classification_report(
    y_teste,
    y_pred_teste,
    target_names=['Sem Diabetes (0)', 'Com Diabetes (1)']
))

### 8.2 Matriz de Confusão

In [ ]:
cm = confusion_matrix(y_teste, y_pred_teste)

fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Sem Diabetes', 'Com Diabetes']
)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Matriz de Confusão — Acurácia: {acuracia_teste*100:.2f}%',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print(f'\nVerdadeiros Negativos (TN): {cm[0][0]}')
print(f'Falsos Positivos       (FP): {cm[0][1]}')
print(f'Falsos Negativos       (FN): {cm[1][0]}')
print(f'Verdadeiros Positivos  (TP): {cm[1][1]}')

### 8.3 Importância das Features

In [ ]:
importancias = pd.Series(
    modelo.feature_importances_,
    index=X.columns
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importancias.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Importância das Features — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importância')
ax.axvline(x=importancias.mean(), color='red', linestyle='--', alpha=0.7, label='Média')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Predição com o Modelo Implantado

Simulando a predição para uma **nova paciente** com dados clínicos.

In [ ]:
nova_paciente = pd.DataFrame([{
    'Gestacoes':        2,
    'Glicose':          138,
    'PressaoSanguinea': 82,
    'EspessuraPele':    25,
    'Insulina':         100,
    'IMC':              31.4,
    'FuncaoPedigree':   0.52,
    'Idade':            35
}])

print('=== Dados da Nova Paciente ===')
print(nova_paciente.to_string(index=False))

nova_paciente_scaled = scaler.transform(nova_paciente)

predicao     = modelo.predict(nova_paciente_scaled)[0]
probabilidade = modelo.predict_proba(nova_paciente_scaled)[0]

print(f'\n=== Resultado da Predição ===')
print(f'Classe predita: {predicao} ({"COM diabetes" if predicao == 1 else "SEM diabetes"})')
print(f'Probabilidade de NÃO ter diabetes: {probabilidade[0]*100:.1f}%')
print(f'Probabilidade de TER diabetes:     {probabilidade[1]*100:.1f}%')

---
## Conclusão

| Etapa | Descrição |
|---|---|
| **Dataset** | Pima Indians Diabetes Database (768 registros, 8 features) |
| **Transformação em colunas** | Substituição de zeros biologicamente impossíveis pela mediana |
| **Transformação em linhas** | Remoção de duplicatas e outliers extremos via IQR |
| **Divisão** | 60% treino / 20% validação / 20% teste (estratificado) |
| **Modelo** | Random Forest Classifier (150 árvores, profundidade máxima 8) |
| **Avaliação** | Acurácia, relatório de classificação e matriz de confusão |

> **Feature mais importante:** Glicose — consistente com a literatura médica,  
> onde o nível de glicose no sangue é o principal marcador diagnóstico do diabetes.